# Apache Arrow Flight

![Arrow Logo](images/arrow.png)


# The Scenario

![City Bikes](images/citybike.jpg)

> Photo by <a href="https://unsplash.com/@jacegrandinetti?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText">Jace & Afsoon</a> on <a href="https://unsplash.com/photos/assorted-color-bicycles-park-beside-blue-rails-near-river-VEXIwDcY1gw?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText">Unsplash</a>

We have our CityBike API, and we need to fetch 1,000,000 records

We need to fetch and process all the records for our analysis.

## With REST API

We have access to a REST API, looks something like this, classic JSON REST API

In [11]:
import httpx2
import polars as pl

REST_API_URL = "https://arrow-flight-rest.fly.dev"
#REST_API_URL = "http://rest:8000"

In [14]:
req = httpx2.get(f"{REST_API_URL}/health")
req.json()

ReadTimeout: The read operation timed out

In [15]:
req = httpx2.get(f"{REST_API_URL}/data/rides/all?num_rows=100", headers={"Authorization": "Bearer pydata_amsterdam"})
pl.from_records(req.json())

ReadTimeout: The read operation timed out

In [4]:
%%timeit -r 2
req = httpx2.get("https://arrow-flight-rest.fly.dev/data/rides/all", headers={"Authorization": "Bearer pydata_amsterdam"})
pl.from_records(req.json())

ReadTimeout: The read operation timed out

## With Arrow Flight

Let's try that again, but with an Arrow Flight server instead

In [5]:
from pyarrow import flight

FLIGHT_SERVER_URL = "grpc+tls://arrow-flight-server.fly.dev:443"
# FLIGHT_SERVER_URL = "grpc://server:7000"

In [6]:
client = flight.connect(FLIGHT_SERVER_URL)

In [7]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))
data = client.do_get(info.endpoints[0].ticket)
pl.from_arrow(data.read_all())

4.14 s ± 178 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


Arrow is a foundational technology in the Data Engineering space. It powers all your favourite tools
 from Spark to Polars, Duckdb and Snowflake. As they say, it's a standard

![XKDC Standards](images/Standards.png)

Why this particular standard matters, is that it solves the problem of interprocess communication
 between different languages and frameworks.

## Process interop

Take the scenario of the Spark UDF in the old days:

![ipc](images/spark_udf.png)

Without a standard for memory layout for the data, we need to introduce glue code between the JVM
Spark Memory and the Pandas Numpy memory layout, usually having to copy the data back and forth.


When we introduce Apache Arrow, both runtimes can share the same memory layout, and we can avoid
the need for any glue code or copying.
![ipc](images/spark_arrow.png)


This is true for any library which uses Arrow, like Duckdb, Pandas and Polars.

![I made this](images/i_made_this.jpg)


## Arrow as the data interchange format
Having Arrow as an universal data interchange format allows libraries to delegate responsibility for their memory layout and compute to 
Arrow and instead focus on their value-adding layer. 

We've seen this before - compilers came along, and gave us all these different optimizations and now we no longer inline statements or unravel loops. 

LLVM and JVM are both examples of the power of separating the layers of a program, so that we can focus on the value-adding part.

So Arrow is great - but what is Arrow Flight then?
